In [ ]:
puntos = [
    ("deposito",         19.63627289905093, -99.200432555112,    0),
    ("club",             19.64454081566007, -99.20515088504385, 35),
    ("escuela",          19.63577572935039, -99.23944300946822, 50),
    ("centro comercial", 19.65901738679457, -99.21075938822268, 60),
    ("planta",           19.64642486258566, -99.18887930812824, 45),
    ("casa amigo",       19.57240666056739, -99.22000711272474, 20),
    ("hospital",         19.59779939115131, -99.18847457071712, 40),
    ("parque",           19.49237842110952, -99.23731457525035, 25),
    ("museo",            19.43112350211384, -99.18824119826257, 30),
]

CAPACIDAD = 120

In [ ]:
import math

def haversine(p1, p2):
    R = 6371.0
    lat1 = math.radians(p1[1]); lon1 = math.radians(p1[2])
    lat2 = math.radians(p2[1]); lon2 = math.radians(p2[2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

In [ ]:
D = []
for i in range(len(puntos)):
    fila = []
    for j in range(len(puntos)):
        fila.append(haversine(puntos[i], puntos[j]))
    D.append(fila)

In [ ]:
demanda_total = 0
for p in puntos:
    demanda_total = demanda_total + p[3]

print(len(D))
print(len(D[0]))
print(demanda_total)
print(demanda_total / CAPACIDAD)
print(haversine(puntos[0], puntos[1]))

9
9
305
2.5416666666666665
1.0437288324120422


In [ ]:
actual = 0
ruta = [0]
visitados = []
carga_restante = CAPACIDAD
total = 0.0
viajes = 1

while len(visitados) < len(puntos) - 1:

    mejor = None
    mejor_dist = float('inf')

    for j in range(len(puntos)):
        if j == 0:
            continue
        if j in visitados:
            continue
        if carga_restante < puntos[j][3]:
            continue
        if D[actual][j] < mejor_dist:
            mejor_dist = D[actual][j]
            mejor = j


    if mejor is None:
        total = total + D[actual][0]
        ruta.append(0)
        actual = 0
        carga_restante = CAPACIDAD
        viajes = viajes + 1
        continue
    total = total + mejor_dist
    ruta.append(mejor)
    visitados.append(mejor)
    carga_restante = carga_restante - puntos[mejor][3]
    actual = mejor

total = total + D[actual][0]
ruta.append(0)

In [ ]:
import itertools
mejor_ruta= None
mejor_total = float('inf')
mejor_viajes = None

for perm in itertools.permutations(range(1, 9)):

    actual = 0
    carga_restante = CAPACIDAD
    d=0.0
    viajes = 1
    camino =  [0]

    for p in perm:
        if puntos [p] [3] > carga_restante:
            d = d + D[actual][p]
            camino.append(p)
            actual = p
            viajes = viajes + 1
            carga_restante = CAPACIDAD
        d = d + D[actual][0]
        camino.append(0)
        carga_restante = carga_restante - puntos[p][3]
        actual = p

     d= d + D[actual][0]
    camino.append(0)
    if d < mejor_total:
         mejor_total = d
        mejor_ruta = camino
        mejor_viajes = viajes

In [ ]:
print(len(mejor_ruta))
print(sorted(set(mejor_ruta)))

suma = 0.0
for k in range(len(mejor_ruta) - 1):
    suma = suma + D[mejor_ruta[k]][mejor_ruta[k+1]]

print(suma)
print(mejor_total)

12
[0, 1, 2, 3, 4, 5, 6, 7, 8]
63.21483381060038
63.21483381060038


In [ ]:
print(mejor_total)
print(mejor_)
print("->".join(puntos[i][0] for i in ruta))

63.21483381060038
3
deposito->club->centro comercial->casa amigo->deposito->planta->hospital->parque->deposito->escuela->museo->deposito


In [ ]:
print(mejor_total)
print(mejor_viajes)
print(sorted(set(mejor_ruta)))
print("->".join(puntos[i][0]for i in mejor_ruta))

suma =0.0
for k in range(len(mejor_ruta)-1):
    suma = suma + D[mejor_ruta[k]][mejor_ruta[k+1]]
print(suma)

63.21483381060038
3
[0, 1, 2, 3, 4, 5, 6, 7, 8]
deposito->club->planta->deposito->escuela->centro comercial->deposito->casa amigo->parque->museo->hospital->deposito
63.21483381060038


# **Optimizador de rutas de reparto**

Calcula la ruta más corta para repartir en 8 puntos desde un depósito, con y sin límite de carga en el camión, y la compara contra la forma intuitiva de planearla.

## Resultados

| |  |
| --- | --- |
| Sin límite de carga, ruta óptima | 56.99 km |
| Con límite de carga, ruta óptima | 63.21 km |
| Con límite de carga, ruta secuencial | 106.45 km |

Las dos conclusiones salen de comparar esos tres números entre sí.

**La restricción es barata. Planear mal es caro.** Ponerle al camión un límite de 120 unidades encarece el reparto 6.22 km, un 10.9%. Planear ese mismo camión con la regla intuitiva lo encarece 43.24 km, un 68%. El mal plan cuesta siete veces más que la restricción

**Y la regla intuitiva empeora más rápido que el problema.** Sin límite de carga, la ruta óptima le gana a la secuencial por 18.4%, con límite de carga, le gana por 40.6%, agregar una restricción no volvió al atajo un poco peor, sino, lo volvió el doble de peor

Los dos óptimos están verificados contra las 40,320 rutas posibles, así que no son estimaciones

## El problema

Si tienes que repartir en 8 lugares y los visitas en el orden que parece lógico, vas a manejar mucho más de lo necesario. Y si además el camión no aguanta todo de un viaje, el problema cambia de forma.

La trampa de la regla intuitiva "voy al que esté más cerca" es que se siente correcta en cada paso y falla en el conjunto. Deja los puntos lejanos para el final y eso obliga a un viaje largo de regreso.

Con límite de carga, esa misma trampa se vuelve peor por una razón nueva: el camión se llena antes de llegar al grupo lejano, y entonces el grupo lejano queda partido entre dos viajes. Cada mitad paga su propio traslado largo.

Las dos soluciones se ven así:

- **Secuencial:** club, centro comercial, casa amigo · planta, hospital, parque · escuela, museo
- **Óptima:** club, planta · escuela, centro comercial · casa amigo, parque, museo, hospital

La secuencial llena el camión desde el primer viaje (115, 110 y 80 unidades). La óptima hace lo contrario: deja el primer viaje ligero (80, 110 y 115), despacha el grupo cercano en dos vueltas cortas, y se guarda capacidad para barrer todo el grupo del sur en un solo recorrido. No parte el grupo lejano, y ahí está el ahorro.

## Cómo funciona

**Datos.** Un depósito y 8 paradas, con coordenadas tomadas de Google Maps y una demanda por parada.
La demanda total es 305 unidades contra una capacidad de 120, así que el reparto no cabe en menos de tres viajes

**Distancias (Haversine).** Calcula la distancia sobre la superficie de la esfera, no usé Pitágoras sobre las coordenadas ya que un grado de longitud no mide lo mismo en todas las latitudes: aquí son unos 105 km y en el ecuador 111

**Matriz de distancias.** Las 81 distancias se calculan una vez y se guardan en una tabla de 9×9, el algoritmo consulta la tabla en lugar de recalcular senos y raíces en cada paso, y queda independiente de cómo se midan las distancias, al agregar la demanda como cuarto dato de cada punto, la función de distancia no necesitó un solo cambio

**Heurística voraz con recargas.** Siempre al punto no visitado más cercano que todavía quepa en el camión. Cuando ya no cabe nada, regresa al depósito, recarga y sigue

**Búsqueda exhaustiva.** Recorre las 40,320 permutaciones de las 8 paradas. Para cada orden, las recargas no son una decisión, el límite de carga las determina solo, porque le conviene regresar al depósito lo más tarde posible

Eso último vale justificarlo, porque de ahí depende que el óptimo sea un óptimo de verdad. Una visita extra al depósito cambia un tramo directo
x → y por x → 0 → y, y sobre una esfera siempre se cumple que D[x][0] + D[0][y] ≥ D[x][y]: el camino por un tercer punto nunca es más corto que el directo. Es la desigualdad del triángulo, y Haversine la respeta, entonces regresar antes de que se llene el camión solo puede sumar kilómetros, y recorrer todos los órdenes regresando únicamente cuando el límite obliga sí cubre la solución óptima. No se queda nada fuera.

## Cómo correrlo

Abre el notebook en Google Colab y corre las celdas de arriba a abajo. No hay nada que instalar: el algoritmo unicamente usa `math` e `itertools`, que vienen con Python.

Para usar otras coordenadas, cambia la lista `puntos` por tus propias coordenadas y demandas, y ajusta `CAPACIDAD`. Google Maps da las coordenadas con clic derecho sobre el mapa

Dos reglas al elegir los datos: la demanda total tiene que superar la capacidad, si no la restricción nunca se activa; y ninguna demanda individual puede pasar de la capacidad, porque entonces esa parada es inalcanzable y el problema no tiene solución

## Verificación

Cada resultado se mide dos veces por caminos distintos. El algoritmo acumula la distancia mientras construye la ruta; después, un bloque aparte recorre la ruta terminada y vuelve a sumar los tramos. Si las dos cifras no coinciden, entonces el acumulador está mal. También se revisa que la ruta ganadora contenga las nueve ubicaciones y que ningún viaje se pase de la capacidad.

Esto no es adorno. Una versión con un error de lógica daba 60.71 km: un número más bajo que el óptimo real, perfectamente creíble al lado de los demás, y producido por un código que corría sin un solo error de sintaxis, la revisión lo detectó en un paso, porque la ruta ganadora solo contenía tres de las nueve ubicaciones

## Limitaciones

**Las distancias son en línea recta.** Haversine mide de punto a punto sobre la esfera, no por calles. El manejo real es mayor, así que todos los números son optimistas, los porcentajes de mejora se sostienen porque las rutas que se comparan se miden igual

**La búsqueda exhaustiva no escala.** Con 8 paradas son 40,320 rutas y tarda menos de un segundo. Con 12 paradas serían 479 millones. Para más puntos hace falta un solver de optimización.

**Un solo vehículo.** El camión recarga en el depósito y vuelve a salir, pero no toma en cuenta si hay flota  y tampoco toma en cuenta ventanas de horario ni costos

## Qué sigue

- **v3:** múltiples vehículos, función objetivo en pesos (combustible + horas + costo fijo por camión), ventanas de horario por parada, distancias reales por calle, y la ruta dibujada sobre un mapa.